# Macis(2024) LSTM Autoencoder — 12년 합본(2014~2026) 재학습 + SE 추출

**목적**: 12년 합본 기준으로 글로벌 LSTM AE 재학습 → 58개국 전체 기간 SE 점수 산출 → `se_scores.parquet` 갱신.

**런타임 설정**: 런타임 → 런타임 유형 변경 → **T4 GPU**

**Config**: base 고정 (`seq_len=30, hidden_dim=64, latent_dim=32, normal_pct=0.75`) — 이전 튜닝에서 채택됨.

**드라이브 구조** (최상단 `My Drive/`, 이미 업로드 완료):
```
conflict-early-warning/
└── input/
    └── processed/
        ├── acled/   ← *.parquet 58개 (~16MB)
        └── gdelt/   ← *.parquet 58개 (~3.3GB)
```

**산출물** (Drive `conflict-early-warning/output/macis_12y/`):
- `model.pt` — 글로벌 모델 (config + per-country scaler 58개)
- `se_scores.parquet` — long-format 테이블 (iso3, date, se_score), LightGBM+SE 피처로 사용
- `case_sanity.csv` — 3국 케이스스터디 sanity check (SDN/PSE/AZE)
- `train_loss.csv` — epoch별 loss 기록

In [ ]:
# ── 1. Drive 마운트 & 경로 설정 ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/conflict-early-warning'
# Drive에 이미 input/processed/ 구조 그대로 업로드돼 있음 (재업로드 불필요)
ACLED_DIR  = f'{DRIVE_ROOT}/input/processed/acled'
GDELT_DIR  = f'{DRIVE_ROOT}/input/processed/gdelt'
OUTPUT_DIR = f'{DRIVE_ROOT}/output/macis_12y'
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.isdir(ACLED_DIR), f'ACLED 폴더 없음: {ACLED_DIR}'
assert os.path.isdir(GDELT_DIR), f'GDELT 폴더 없음: {GDELT_DIR}'

print(f'ACLED 파일 수: {len(os.listdir(ACLED_DIR))} (기대 58)')
print(f'GDELT 파일 수: {len(os.listdir(GDELT_DIR))} (기대 58)')
print(f'출력 디렉토리: {OUTPUT_DIR}')

In [ ]:
# ── 2. 패키지 import ─────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'pyarrow', '-q'], capture_output=True)

import time
import json
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── 3. 데이터 로딩 (src/model/macis_data.py 인라인) ────────────
GDELT_MENTIONS_THRESHOLD = 10

_ACLED_EVENT_TYPES = {
    'Battles': 'battles',
    'Explosions/Remote violence': 'explosions',
    'Violence against civilians': 'vac',
    'Riots': 'riots',
    'Protests': 'protests',
    'Strategic developments': 'strategic',
}

FEATURE_COLS = [
    'battles', 'explosions', 'vac', 'riots', 'protests', 'strategic',
    'fatalities', 'violence',
    'gdelt_n_events', 'gdelt_goldstein_mean', 'gdelt_tone_mean',
    'gdelt_quad1_n', 'gdelt_quad2_n', 'gdelt_quad3_n', 'gdelt_quad4_n',
]

def load_acled_daily(iso3):
    path = f'{ACLED_DIR}/{iso3}.parquet'
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_parquet(path)
    df['event_date'] = pd.to_datetime(df['event_date'], utc=True).dt.normalize()
    rows = []
    for date, grp in df.groupby('event_date'):
        row = {'date': date, 'fatalities': int(grp['fatalities'].sum())}
        for raw_type, col in _ACLED_EVENT_TYPES.items():
            row[col] = int((grp['event_type'] == raw_type).sum())
        excessive = 0
        if 'sub_event_type' in grp.columns:
            excessive = int((grp['sub_event_type'] == 'Excessive force against protesters').sum())
        row['violence'] = row['battles'] + row['explosions'] + row['vac'] + row['riots'] + excessive
        rows.append(row)
    if not rows:
        return pd.DataFrame()
    daily = pd.DataFrame(rows).set_index('date').sort_index()
    full_idx = pd.date_range(daily.index.min(), daily.index.max(), freq='D', tz='UTC')
    return daily.reindex(full_idx, fill_value=0)

def load_gdelt_daily(iso3):
    path = f'{GDELT_DIR}/{iso3}.parquet'
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_parquet(path)
    df['event_date'] = pd.to_datetime(df['event_date'], utc=True).dt.normalize()
    df = df[df['NumMentions'] >= GDELT_MENTIONS_THRESHOLD].copy()
    if df.empty:
        return pd.DataFrame()
    daily = df.groupby('event_date').agg(
        gdelt_n_events=('GLOBALEVENTID', 'count'),
        gdelt_goldstein_mean=('GoldsteinScale', 'mean'),
        gdelt_tone_mean=('AvgTone', 'mean'),
    )
    for qc in [1, 2, 3, 4]:
        qdf = df[df['QuadClass'] == qc].groupby('event_date').size().rename(f'gdelt_quad{qc}_n')
        daily = daily.join(qdf, how='left')
    quad_cols = [f'gdelt_quad{qc}_n' for qc in [1,2,3,4]]
    daily[quad_cols] = daily[quad_cols].fillna(0)
    full_idx = pd.date_range(daily.index.min(), daily.index.max(), freq='D', tz='UTC')
    daily = daily.reindex(full_idx)
    daily['gdelt_n_events'] = daily['gdelt_n_events'].fillna(0)
    daily['gdelt_goldstein_mean'] = daily['gdelt_goldstein_mean'].fillna(0)
    daily['gdelt_tone_mean'] = daily['gdelt_tone_mean'].fillna(0)
    daily[quad_cols] = daily[quad_cols].fillna(0)
    return daily

def load_macis_features(iso3):
    acled = load_acled_daily(iso3)
    gdelt = load_gdelt_daily(iso3)
    features = acled.join(gdelt, how='inner').sort_index()
    target = features['fatalities'].copy()
    return features, target

print('데이터 로딩 함수 정의 완료')

In [ ]:
# ── 4. LSTM Autoencoder (src/model/autoencoder_model.py 인라인) ─
class _LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, hidden_dim=64, latent_dim=32):
        super().__init__()
        self.encoder_lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.encoder_fc   = nn.Linear(hidden_dim, latent_dim)
        self.decoder_fc   = nn.Linear(latent_dim, hidden_dim)
        self.decoder_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.decoder_out  = nn.Linear(hidden_dim, n_features)

    def forward(self, x):
        bs, seq, _ = x.shape
        _, (h_n, _) = self.encoder_lstm(x)
        latent = self.encoder_fc(h_n[-1])
        dec_input = self.decoder_fc(latent).unsqueeze(1).repeat(1, seq, 1)
        dec_out, _ = self.decoder_lstm(dec_input)
        return self.decoder_out(dec_out)


class LSTMAutoencoderModel:
    def __init__(self, n_features, seq_len=30, hidden_dim=64, latent_dim=32,
                 lr=1e-3, epochs=300, batch_size=128, patience=20):
        self.n_features = n_features
        self.seq_len = seq_len
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience
        self.scaler = StandardScaler()
        self.per_country_scalers = {}
        self.model = None

    def _to_windows(self, X):
        T, F = X.shape
        n = T - self.seq_len + 1
        w = np.zeros((n, self.seq_len, F), dtype=np.float32)
        for i in range(n):
            w[i] = X[i:i+self.seq_len]
        return w

    def fit_global(self, country_data, verbose=True):
        all_windows = []
        for iso3, X_normal in country_data.items():
            if len(X_normal) < self.seq_len + 1:
                if verbose:
                    print(f'  [{iso3}] 데이터 부족 ({len(X_normal)}일) — 건너뜀')
                continue
            s = StandardScaler()
            X_scaled = s.fit_transform(X_normal.values).astype(np.float32)
            self.per_country_scalers[iso3] = s
            all_windows.append(self._to_windows(X_scaled))
        if not all_windows:
            raise ValueError('데이터 없음')
        combined = np.concatenate(all_windows, axis=0)
        if verbose:
            print(f'  글로벌 학습 데이터: {len(combined):,} 윈도우 ({len(country_data)}개국)')

        dataset = torch.tensor(combined, dtype=torch.float32)
        loader  = torch.utils.data.DataLoader(
            dataset, batch_size=self.batch_size, shuffle=True)
        self.model = _LSTMAutoencoder(
            self.n_features, self.hidden_dim, self.latent_dim).to(DEVICE)
        opt  = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        crit = nn.MSELoss()

        best_loss, wait, best_state = float('inf'), 0, None
        losses = []
        self.model.train()
        for epoch in range(self.epochs):
            t0 = time.time()
            ep_loss = 0.0
            for batch in loader:
                batch = batch.to(DEVICE)
                opt.zero_grad()
                recon = self.model(batch)
                loss  = crit(recon, batch)
                loss.backward()
                opt.step()
                ep_loss += loss.item() * len(batch)
            ep_loss /= len(dataset)
            losses.append(ep_loss)
            if ep_loss < best_loss - 1e-6:
                best_loss, wait = ep_loss, 0
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
            else:
                wait += 1
                if wait >= self.patience:
                    if verbose:
                        print(f'  Early stop @ epoch {epoch+1} (best loss={best_loss:.6f})')
                    break
            if verbose and (epoch+1) % 10 == 0:
                print(f'  Epoch {epoch+1:3d}: loss={ep_loss:.6f}  ({time.time()-t0:.1f}s)')
        if best_state:
            self.model.load_state_dict(best_state)
        return losses

    def reconstruction_error(self, X, iso3=None):
        if self.model is None:
            raise RuntimeError('fit_global() 먼저 호출하세요')
        scaler = self.per_country_scalers.get(iso3) if iso3 else self.scaler
        X_scaled = scaler.transform(X.values).astype(np.float32)
        windows = self._to_windows(X_scaled)
        self.model.eval()
        ses = []
        with torch.no_grad():
            for i in range(0, len(windows), self.batch_size):
                b = torch.tensor(windows[i:i+self.batch_size]).to(DEVICE)
                r = self.model(b)
                ses.append(((b[:,-1,:] - r[:,-1,:])**2).mean(dim=1).cpu().numpy())
        full = np.full(len(X), np.nan)
        full[self.seq_len-1:] = np.concatenate(ses)
        return pd.Series(full, index=X.index, name='se')

    def save(self, path):
        pcs = {iso3: {'mean': s.mean_, 'scale': s.scale_}
               for iso3, s in self.per_country_scalers.items()}
        torch.save({
            'model_state': self.model.state_dict() if self.model else None,
            'per_country_scalers': pcs,
            'config': {
                'n_features': self.n_features, 'seq_len': self.seq_len,
                'hidden_dim': self.hidden_dim, 'latent_dim': self.latent_dim,
            },
        }, path)
        print(f'  저장: {path}')

print('Autoencoder 클래스 정의 완료')

In [ ]:
# ── 5. 설정: base config 고정 ────────────────────────────────
CONFIG = {
    'seq_len':     30,
    'hidden_dim':  64,
    'latent_dim':  32,
    'normal_pct':  0.75,
    'epochs':      300,
    'batch_size':  256,    # 12년치 윈도우 많아서 batch 높임 (T4 16GB 안전)
    'lr':          1e-3,
    'patience':    20,
}

# 케이스 홀드아웃 (이벤트일 기준 D-240일 이후는 학습에서 제외 — sanity check용)
CASE_STUDIES = {
    'SDN': {'label': 'Sudan Civil War',          'event_date': '2023-04-15'},
    'PSE': {'label': 'Gaza Oct7 War',            'event_date': '2023-10-07'},
    'AZE': {'label': 'Azerbaijan Karabakh Op.',  'event_date': '2023-09-19'},
}
HOLDOUT_DAYS = 240

print('Config:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# ── 6. Normal days 수집 (58개국, 12년치) ───────────────────────
def collect_normal_days(normal_percentile=0.75, seq_len=30):
    country_data = {}
    all_iso3 = sorted([f.replace('.parquet','') for f in os.listdir(ACLED_DIR)
                       if f.endswith('.parquet') and not f.startswith('.')])
    for iso3 in all_iso3:
        try:
            features, _ = load_macis_features(iso3)
        except Exception as e:
            print(f'  [{iso3}] 로딩 실패: {e}')
            continue
        if features.empty or len(features) < seq_len + 1:
            continue
        # 케이스 국가: D-240일 이전만 학습에 사용
        if iso3 in CASE_STUDIES:
            cutoff = pd.Timestamp(CASE_STUDIES[iso3]['event_date'], tz='UTC') - pd.Timedelta(days=HOLDOUT_DAYS)
            features = features[features.index < cutoff]
        if len(features) < seq_len + 1:
            continue
        fat = features['fatalities']
        thr = fat.quantile(normal_percentile)
        mask = fat < thr if thr > 0 else pd.Series(True, index=features.index)
        feat_cols = [c for c in FEATURE_COLS if c in features.columns]
        X_normal = features.loc[mask, feat_cols]
        if len(X_normal) >= seq_len + 1:
            country_data[iso3] = X_normal
    total = sum(len(v) for v in country_data.values())
    print(f'  {len(country_data)}개국, {total:,}일 normal data 수집')
    return country_data

country_data = collect_normal_days(
    normal_percentile=CONFIG['normal_pct'],
    seq_len=CONFIG['seq_len'],
)
n_features = len(next(iter(country_data.values())).columns)
print(f'  n_features: {n_features}')
print(f'  국가별 데이터 분포 (상위 5):')
for iso3 in list(country_data)[:5]:
    print(f'    {iso3}: {len(country_data[iso3]):,}일, {country_data[iso3].index.min().date()} ~ {country_data[iso3].index.max().date()}')

In [ ]:
# ── 7. 글로벌 학습 ───────────────────────────────────────────
ae = LSTMAutoencoderModel(
    n_features  = n_features,
    seq_len     = CONFIG['seq_len'],
    hidden_dim  = CONFIG['hidden_dim'],
    latent_dim  = CONFIG['latent_dim'],
    epochs      = CONFIG['epochs'],
    batch_size  = CONFIG['batch_size'],
    lr          = CONFIG['lr'],
    patience    = CONFIG['patience'],
)

t0 = time.time()
losses = ae.fit_global(country_data, verbose=True)
elapsed = time.time() - t0
print(f'\n학습 완료: {elapsed/60:.1f}분 ({len(losses)} epochs)')

# loss 곡선 저장
pd.DataFrame({'epoch': range(1, len(losses)+1), 'train_loss': losses}).to_csv(
    f'{OUTPUT_DIR}/train_loss.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(losses)+1), losses, marker='.', markersize=3)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Macis LSTM AE 학습 경과 (12년 합본, base config)')
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/train_loss.png', dpi=120)
plt.close(fig)
print(f'  loss 곡선: {OUTPUT_DIR}/train_loss.png')

In [ ]:
# ── 8. 모델 저장 ─────────────────────────────────────────────
MODEL_PATH = f'{OUTPUT_DIR}/model.pt'
ae.save(MODEL_PATH)

# config json 별도 저장 (로컬 복원 시 참고용)
with open(f'{OUTPUT_DIR}/config.json', 'w') as f:
    json.dump({
        **CONFIG,
        'n_features': n_features,
        'data_range': '2014-2026 (12년 합본)',
        'n_countries_trained': len(ae.per_country_scalers),
        'total_windows': sum(len(v) - CONFIG['seq_len'] + 1 for v in country_data.values()),
    }, f, indent=2, ensure_ascii=False)
print(f'  config: {OUTPUT_DIR}/config.json')

In [ ]:
# ── 9. 전체 국가×일 SE 점수 산출 ─────────────────────────────
all_iso3 = sorted([f.replace('.parquet','') for f in os.listdir(ACLED_DIR)
                   if f.endswith('.parquet') and not f.startswith('.')])
print(f'SE 산출 대상: {len(all_iso3)}개국')

rows = []
failed = []
for i, iso3 in enumerate(all_iso3, 1):
    try:
        features, _ = load_macis_features(iso3)
    except Exception as e:
        print(f'  [{i:2d}/{len(all_iso3)}] {iso3}: 로딩 실패 ({e})')
        failed.append(iso3)
        continue
    if features.empty or len(features) < CONFIG['seq_len']:
        print(f'  [{i:2d}/{len(all_iso3)}] {iso3}: 데이터 부족 ({len(features)}일)')
        failed.append(iso3)
        continue

    feat_cols = [c for c in FEATURE_COLS if c in features.columns]
    X = features[feat_cols]

    # 글로벌 학습에 포함되지 않은 국가는 전체 데이터로 임시 scaler 학습
    if iso3 not in ae.per_country_scalers:
        s = StandardScaler()
        s.fit(X.values)
        ae.per_country_scalers[iso3] = s
        print(f'  [{iso3}] 신규 scaler (글로벌 학습 미포함)')

    se = ae.reconstruction_error(X, iso3=iso3)
    df = se.reset_index()
    df.columns = ['date', 'se_score']
    df.insert(0, 'iso3', iso3)
    rows.append(df)
    n_valid = se.notna().sum()
    print(f'  [{i:2d}/{len(all_iso3)}] {iso3}: {len(se):,}일 (valid {n_valid:,}, NaN {se.isna().sum()})')

se_df = pd.concat(rows, ignore_index=True)
se_df['date'] = pd.to_datetime(se_df['date'], utc=True)
SE_PATH = f'{OUTPUT_DIR}/se_scores.parquet'
se_df.to_parquet(SE_PATH, index=False)

print(f'\n=== SE 테이블 저장 완료 ===')
print(f'  경로: {SE_PATH}')
print(f'  행수: {len(se_df):,}')
print(f'  국가 수: {se_df["iso3"].nunique()}')
print(f'  기간: {se_df["date"].min().date()} ~ {se_df["date"].max().date()}')
print(f'  NaN 비율: {se_df["se_score"].isna().mean()*100:.2f}%')
print(f'  se_score 통계: mean={se_df["se_score"].mean():.4f}, median={se_df["se_score"].median():.4f}, p95={se_df["se_score"].quantile(0.95):.4f}, max={se_df["se_score"].max():.4f}')
if failed:
    print(f'  실패 국가: {failed}')

In [ ]:
# ── 10. Sanity check — 3국 케이스스터디 ROC AUC ───────────────
LABEL_NORMAL, LABEL_PRE_UNREST, LABEL_UNREST = 0, 1, 2
TRAIN_VAL_SPLIT_DAYS, VAL_TEST_SPLIT_DAYS = 240, 120

def label_days(target, threshold, pre_unrest_n, force_unrest_dates=None):
    labels = pd.Series(LABEL_NORMAL, index=target.index, dtype=int)
    unrest_days = target.index[target > threshold]
    labels[unrest_days] = LABEL_UNREST
    if force_unrest_dates:
        for d in force_unrest_dates:
            if d in labels.index:
                labels[d] = LABEL_UNREST
    for uday in labels.index[labels == LABEL_UNREST]:
        for delta in range(1, pre_unrest_n + 1):
            pre_day = uday - pd.Timedelta(days=delta)
            if pre_day in labels.index and labels[pre_day] == LABEL_NORMAL:
                labels[pre_day] = LABEL_PRE_UNREST
    return labels

def grid_search_quick(target, train_end, val_end, se):
    val_mask = (se.index > train_end) & (se.index <= val_end)
    val_idx  = se.index[val_mask]
    p98, p995 = target.quantile(0.98), target.quantile(0.995)
    thresholds = [p98] if p98 >= p995 else list(np.linspace(p98, p995, 5))
    best = (None, None, -1.0)
    for thr in thresholds:
        for n in range(30, 91, 7):
            labels = label_days(target, thr, n)
            val_labels = labels[val_idx]
            val_se     = se[val_idx]
            if val_labels.isin([LABEL_PRE_UNREST]).sum() == 0:
                continue
            mask = val_labels != LABEL_UNREST
            y_true  = (val_labels[mask] == LABEL_PRE_UNREST).astype(int)
            y_score = val_se[mask].values
            if len(y_true.unique()) < 2:
                continue
            try:
                auc = roc_auc_score(y_true, y_score)
                if auc > best[2]:
                    best = (thr, n, auc)
            except Exception:
                continue
    return best

sanity_rows = []
for iso3, case in CASE_STUDIES.items():
    event_date = pd.Timestamp(case['event_date'], tz='UTC')
    features, _ = load_macis_features(iso3)
    features = features[features.index <= event_date]
    target   = features['fatalities']
    feat_cols = [c for c in FEATURE_COLS if c in features.columns]
    X = features[feat_cols]

    se = ae.reconstruction_error(X, iso3=iso3)
    train_end = event_date - pd.Timedelta(days=TRAIN_VAL_SPLIT_DAYS)
    val_end   = event_date - pd.Timedelta(days=VAL_TEST_SPLIT_DAYS)
    best_thr, best_n, val_auc = grid_search_quick(target, train_end, val_end, se)
    if best_thr is None:
        sanity_rows.append({'iso3': iso3, 'label': case['label'], 'val_auc': float('nan'), 'test_auc': float('nan')})
        print(f'  {iso3}: grid search 실패')
        continue
    final_labels = label_days(target, best_thr, best_n, [event_date])
    test_mask = (X.index > val_end) & (X.index <= event_date)
    test_se   = se[test_mask].dropna()
    test_lb   = final_labels[test_se.index]
    mask_b    = test_lb != LABEL_UNREST
    y_true    = (test_lb[mask_b] == LABEL_PRE_UNREST).astype(int)
    y_score   = test_se[mask_b].values
    test_auc  = roc_auc_score(y_true, y_score) if len(y_true.unique()) >= 2 else float('nan')
    sanity_rows.append({
        'iso3': iso3, 'label': case['label'],
        'best_threshold': best_thr, 'best_pre_unrest_n': best_n,
        'val_auc': val_auc, 'test_auc': test_auc,
    })
    print(f'  {iso3}: val_auc={val_auc:.4f}, test_auc={test_auc:.4f}, thr={best_thr:.1f}, n={best_n}')

sanity_df = pd.DataFrame(sanity_rows)
sanity_df.to_csv(f'{OUTPUT_DIR}/case_sanity.csv', index=False)
print(f'\nsanity check 저장: {OUTPUT_DIR}/case_sanity.csv')
print(f'  mean test AUC: {sanity_df["test_auc"].mean():.4f}  (이전 1.5y base: 0.541)')

In [ ]:
# ── 11. 산출물 최종 확인 ─────────────────────────────────────
print('=== 출력 파일 목록 ===')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}')
    print(f'  {f}: {size:,} bytes ({size/1024/1024:.2f} MB)')

print(f'\n로컬 다운로드 안내:')
print(f'  Drive: {OUTPUT_DIR.replace("/content/drive/MyDrive", "My Drive")}')
print(f'  로컬 배치 경로:')
print(f'    model.pt           →  output/models/macis_global/model.pt   (덮어쓰기)')
print(f'    se_scores.parquet  →  input/processed/features/se_scores.parquet  (덮어쓰기)')
print(f'    config.json        →  output/models/macis_global/config.json')
print(f'    case_sanity.csv    →  output/tuning/macis_12y_sanity.csv  (참고용)')